### Restart and Run All Cells

In [2]:
import pandas as pd
from datetime import date, timedelta, datetime
from sqlalchemy import create_engine, text
engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()
year = 2025
quarter = 3
today = date.today()
today_str = today.strftime("%Y-%m-%d")
current_time = datetime.now()
current_time

datetime.datetime(2025, 11, 30, 11, 26, 46, 52658)

In [3]:
cols = 'name year quarter q_amt_c q_amt_p inc_profit percent'.split()

format_dict = {
                'q_amt':'{:,}','q_amt_c':'{:,}','q_amt_p':'{:,}','inc_profit':'{:,}',
                'yoy_gain':'{:,}','acc_gain':'{:,}',    
                'q_eps':'{:.4f}','y_eps':'{:.4f}','aq_eps':'{:.4f}','ay_eps':'{:.4f}',
                'percent':'{:.2f}%'
              }

In [4]:
sql = """
SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = %s AND quarter <= %s) 
OR (year = %s-1 AND quarter >= %s+1)
ORDER BY year DESC, quarter DESC"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = 2025 AND quarter <= 3) 
OR (year = 2025-1 AND quarter >= 3+1)
ORDER BY year DESC, quarter DESC


In [5]:
dfc = pd.read_sql(sql, conlt)
dfc["Counter"] = 1
dfc_grp = dfc.groupby(["name"], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp["Counter"] == 4]
dfc_grp.sample(5).style.format(format_dict)

,name,year,quarter,q_amt,Counter
115,MCS,8099,10,"1,043,387",4
206,TTLPF,8099,10,"291,334",4
72,GGC,8099,10,"-609,000",4
175,STGT,8099,10,"951,945",4
170,SPRC,8099,10,"1,641,889",4


In [6]:
sql = """
SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = %s-1 AND quarter <= %s) 
OR (year = %s-2 AND quarter >= %s+1)
ORDER BY year DESC, quarter DESC
"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = 2025-1 AND quarter <= 3) 
OR (year = 2025-2 AND quarter >= 3+1)
ORDER BY year DESC, quarter DESC



In [7]:
dfp = pd.read_sql(sql, conlt)
dfp["Counter"] = 1
dfp_grp = dfp.groupby(["name"], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp["Counter"] == 4]
dfp_grp.sample(5).style.format(format_dict)

,name,year,quarter,q_amt,Counter
59,DIF,8095,10,"7,710,754",4
208,TTB,8095,10,"20,785,524",4
113,M,8095,10,"1,597,148",4
211,TU,8095,10,"-13,417,443",4
87,IMH,8095,10,"-34,387",4


In [8]:
dfm = pd.merge(dfc_grp, dfp_grp, on="name", suffixes=(["_c", "_p"]), how="inner")
dfm["inc_profit"] = dfm["q_amt_c"] - dfm["q_amt_p"]
dfm["percent"] = round(dfm["inc_profit"] / abs(dfm["q_amt_p"]) * 100, 2)
dfm["year"] = year
dfm["quarter"] = "Q" + str(quarter)
df_percent = dfm[cols]
df_percent.sample(5).style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
38,CBG,2025,Q3,"2,959,799","2,709,312","250,487",9.25%
150,SKR,2025,Q3,"619,344","844,166","-224,822",-26.63%
198,WHART,2025,Q3,"2,164,119","2,247,931","-83,812",-3.73%
104,MEGA,2025,Q3,"1,974,367","1,846,238","128,129",6.94%
18,BAM,2025,Q3,"2,217,298","1,538,803","678,495",44.09%


In [9]:
# Create the SQL query with parameter binding
sql = text("DELETE FROM yr_profits WHERE year = :year AND quarter = :quarter")

# Execute the query with parameters
params = {'year': year, 'quarter': f'Q{quarter}'}
rp = conlt.execute(sql, params)

# Print the number of rows affected
print("Rows deleted:", rp.rowcount)

Rows deleted: 124


In [10]:
sql = "SELECT name, id FROM tickers"
tickers = pd.read_sql(sql, conlt)
df_ins = pd.merge(df_percent, tickers, on="name", how="inner")
rcds = df_ins.values.tolist()
len(rcds)

202

In [11]:
# Convert DataFrame to list of records
rcds = df_ins.values.tolist()

# Define column names in the same order as values
columns = ['name', 'year', 'quarter', 'latest_amt', 'previous_amt', 'inc_amt', 'inc_pct', 'ticker_id']

# SQL insert statement with named parameters
sql = text("""
    INSERT INTO yr_profits 
    (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id)
    VALUES (:name, :year, :quarter, :latest_amt, :previous_amt, :inc_amt, :inc_pct, :ticker_id)
""")

try:
    # Execute inserts
    for rcd in rcds:
        # Convert list to dictionary
        params = dict(zip(columns, rcd))
        conlt.execute(sql, params)
except Exception as e:
    raise e

### End of loop

In [13]:
criteria_1 = df_ins.q_amt_c > 440_000
df_ins.loc[criteria_1, cols].style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q3,"7,719,299","2,443,671","5,275,628",215.89%
1,ACE,2025,Q3,"839,069","750,207","88,862",11.84%
2,ADVANC,2025,Q3,"42,863,183","32,818,977","10,044,206",30.60%
3,AH,2025,Q3,"745,761","764,738","-18,977",-2.48%
5,AIMIRT,2025,Q3,"830,452","698,124","132,328",18.95%
6,AIT,2025,Q3,"590,649","584,604","6,045",1.03%
8,AMATA,2025,Q3,"3,130,606","2,142,951","987,655",46.09%
10,AOT,2025,Q3,"18,534,412","18,342,282","192,130",1.05%
11,AP,2025,Q3,"4,317,587","5,062,056","-744,469",-14.71%
12,ASIAN,2025,Q3,"647,824","815,672","-167,848",-20.58%


In [14]:
criteria_2 = df_ins.q_amt_p > 400_000
df_ins.loc[criteria_2, cols].style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q3,"7,719,299","2,443,671","5,275,628",215.89%
1,ACE,2025,Q3,"839,069","750,207","88,862",11.84%
2,ADVANC,2025,Q3,"42,863,183","32,818,977","10,044,206",30.60%
3,AH,2025,Q3,"745,761","764,738","-18,977",-2.48%
5,AIMIRT,2025,Q3,"830,452","698,124","132,328",18.95%
6,AIT,2025,Q3,"590,649","584,604","6,045",1.03%
8,AMATA,2025,Q3,"3,130,606","2,142,951","987,655",46.09%
10,AOT,2025,Q3,"18,534,412","18,342,282","192,130",1.05%
11,AP,2025,Q3,"4,317,587","5,062,056","-744,469",-14.71%
12,ASIAN,2025,Q3,"647,824","815,672","-167,848",-20.58%


In [15]:
criteria_3 = df_ins.percent > 10.00
df_ins.loc[criteria_3, cols].style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q3,"7,719,299","2,443,671","5,275,628",215.89%
1,ACE,2025,Q3,"839,069","750,207","88,862",11.84%
2,ADVANC,2025,Q3,"42,863,183","32,818,977","10,044,206",30.60%
4,AIE,2025,Q3,"168,287","138,452","29,835",21.55%
5,AIMIRT,2025,Q3,"830,452","698,124","132,328",18.95%
8,AMATA,2025,Q3,"3,130,606","2,142,951","987,655",46.09%
9,ANAN,2025,Q3,"139,548","120,459","19,089",15.85%
16,AWC,2025,Q3,"6,381,684","5,348,602","1,033,082",19.31%
17,BA,2025,Q3,"3,648,593","2,910,875","737,718",25.34%
18,BAM,2025,Q3,"2,217,298","1,538,803","678,495",44.09%


In [16]:
final_criteria = criteria_1 & criteria_2 & criteria_3
df_ins.loc[final_criteria, cols].sort_values(by=["percent"], ascending=[False]).style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
70,GULF,2025,Q3,"96,429,425","19,032,052","77,397,373",406.67%
168,TFG,2025,Q3,"7,170,526","1,427,538","5,742,988",402.30%
186,TTA,2025,Q3,"2,303,219","462,798","1,840,421",397.67%
0,3BBIF,2025,Q3,"7,719,299","2,443,671","5,275,628",215.89%
140,SCC,2025,Q3,"17,254,554","5,719,716","11,534,838",201.67%
111,OR,2025,Q3,"12,224,912","4,843,875","7,381,037",152.38%
36,BLA,2025,Q3,"7,271,185","3,027,476","4,243,709",140.17%
113,OSP,2025,Q3,"3,541,517","1,504,018","2,037,499",135.47%
103,MCS,2025,Q3,"1,043,387","471,000","572,387",121.53%
105,MINT,2025,Q3,"9,687,380","5,102,760","4,584,620",89.85%


In [17]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q3,"7,719,299","2,443,671","5,275,628",215.89%
1,ACE,2025,Q3,"839,069","750,207","88,862",11.84%
2,ADVANC,2025,Q3,"42,863,183","32,818,977","10,044,206",30.60%
5,AIMIRT,2025,Q3,"830,452","698,124","132,328",18.95%
8,AMATA,2025,Q3,"3,130,606","2,142,951","987,655",46.09%
16,AWC,2025,Q3,"6,381,684","5,348,602","1,033,082",19.31%
17,BA,2025,Q3,"3,648,593","2,910,875","737,718",25.34%
18,BAM,2025,Q3,"2,217,298","1,538,803","678,495",44.09%
21,BBL,2025,Q3,"48,651,438","43,669,640","4,981,798",11.41%
32,BGRIM,2025,Q3,"1,968,731","1,233,128","735,603",59.65%


In [18]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q3,"7,719,299","2,443,671","5,275,628",215.89%
1,ACE,2025,Q3,"839,069","750,207","88,862",11.84%
2,ADVANC,2025,Q3,"42,863,183","32,818,977","10,044,206",30.60%
5,AIMIRT,2025,Q3,"830,452","698,124","132,328",18.95%
8,AMATA,2025,Q3,"3,130,606","2,142,951","987,655",46.09%
16,AWC,2025,Q3,"6,381,684","5,348,602","1,033,082",19.31%
17,BA,2025,Q3,"3,648,593","2,910,875","737,718",25.34%
18,BAM,2025,Q3,"2,217,298","1,538,803","678,495",44.09%
21,BBL,2025,Q3,"48,651,438","43,669,640","4,981,798",11.41%
32,BGRIM,2025,Q3,"1,968,731","1,233,128","735,603",59.65%


In [19]:
conlt.commit()
conlt.close()

In [20]:
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2025:11:30 11:26:46
